# Assignment 3 — End-to-End Portal Design
## Student Management Portal (Python backend + Flask website)
**Team:** 25L-5308 · 25L-5291

This notebook runs the portal's backend (`backend.py`) and shows sample test runs.
It demonstrates **functions (`def`)**, **loops (`for`, `while`)**, **file handling (JSON)** and **exception handling (`try / except / finally`)**.
The website (`app.py`) is a Flask frontend that calls the same backend functions.

## 1. Setup
The notebook uses its own sandbox data folder so test runs never touch real portal data.

In [1]:
import os, shutil, json
import backend

SANDBOX = "notebook_data"
shutil.rmtree(SANDBOX, ignore_errors=True)          # start every run clean
backend.DATA_DIR = SANDBOX
backend.STUDENTS_FILE = os.path.join(SANDBOX, "students.json")
backend.USERS_FILE = os.path.join(SANDBOX, "users.json")
print("Sandbox ready:", SANDBOX)

Sandbox ready: notebook_data


## 2. File handling — startup with no data
`load_json()` returns an empty list when the file doesn't exist (handles `FileNotFoundError`), so the portal starts cleanly on first run.

In [2]:
print("Students on first start:", backend.load_records())
print("Users file created with default admin:", [u["username"] for u in backend.load_users()])

Students on first start: []
Users file created with default admin: ['admin']


## 3. `user_login()` — authentication
Passwords are stored as salted SHA-256 hashes, never as plain text.

In [3]:
print("Correct login  ->", backend.user_login("admin", "admin123"))
print("Wrong password ->", backend.user_login("admin", "wrong"))
print("Unknown user   ->", backend.user_login("nobody", "admin123"))

Correct login  -> {'username': 'admin', 'role': 'admin'}
Wrong password -> None
Unknown user   -> None


## 4. `add_record()` — adding students (loop + persistence)

In [4]:
sample = [
    ("25L-5308", "Ayesha Khan",  "Computer Science",     3, 88),
    ("25L-5291", "Bilal Ahmed",  "Computer Science",     3, 72.5),
    ("25L-5101", "Sara Malik",   "Software Engineering", 2, 91),
    ("25L-5133", "Usman Raza",   "Electrical Engineering", 4, 48),
    ("25L-5160", "Ali Hassan",   "Business Administration", 1, 36),
    ("25L-5172", "Hira Noor",    "Law",                  5, 95),
]
for roll, name, dept, sem, marks in sample:
    ok, msgs = backend.add_record({"roll_no": roll, "name": name, "department": dept,
                                   "semester": sem, "marks": marks})
    print(ok, "|", msgs[0])

True | Added Ayesha Khan (25L-5308).
True | Added Bilal Ahmed (25L-5291).
True | Added Sara Malik (25L-5101).
True | Added Usman Raza (25L-5133).
True | Added Ali Hassan (25L-5160).
True | Added Hira Noor (25L-5172).


## 5. Error handling — invalid input and duplicates
`ValueError` from `int()` / `float()` is caught inside `validate_record()`; the portal reports every problem instead of crashing.

In [5]:
tests = {
  "text in marks":    {"roll_no": "T1", "name": "Test", "department": "Law", "semester": "2", "marks": "abc"},
  "marks over 100":   {"roll_no": "T2", "name": "Test", "department": "Law", "semester": "2", "marks": "150"},
  "bad semester":     {"roll_no": "T3", "name": "Test", "department": "Law", "semester": "twelve", "marks": "70"},
  "empty name":       {"roll_no": "T4", "name": "  ",  "department": "Law", "semester": "2", "marks": "70"},
  "duplicate roll no":{"roll_no": "25l-5308", "name": "Copy", "department": "Law", "semester": "2", "marks": "70"},
}
for label, data in tests.items():
    ok, msgs = backend.add_record(data)
    print(f"{label:18} -> saved={ok}  {msgs}")

text in marks      -> saved=False  ['Marks must be a number.']
marks over 100     -> saved=False  ['Marks must be between 0 and 100.']
bad semester       -> saved=False  ['Semester must be a whole number.']
empty name         -> saved=False  ['Name is required.']
duplicate roll no  -> saved=False  ['Roll number 25L-5308 already exists.']


## 6. `search_record()` — filtering with a `for` loop

In [6]:
def show(rows):
    for r in rows:
        print(f"  {r['roll_no']:<9}{r['name']:<14}{r['department']:<24}sem {r['semester']}  {r['marks']:>5}  {r['grade']}")

print("Search name contains 'a':");            show(backend.search_record("a", "name"))
print("\nSearch department 'law':");           show(backend.search_record("law", "department"))
print("\nSearch roll_no '5291':");             show(backend.search_record("5291", "roll_no"))
print("\nNo match:", backend.search_record("zzz", "name"))

Search name contains 'a':
  25L-5308 Ayesha Khan   Computer Science        sem 3   88.0  A
  25L-5291 Bilal Ahmed   Computer Science        sem 3   72.5  B
  25L-5101 Sara Malik    Software Engineering    sem 2   91.0  A
  25L-5133 Usman Raza    Electrical Engineering  sem 4   48.0  D
  25L-5160 Ali Hassan    Business Administration sem 1   36.0  F
  25L-5172 Hira Noor     Law                     sem 5   95.0  A

Search department 'law':
  25L-5172 Hira Noor     Law                     sem 5   95.0  A

Search roll_no '5291':
  25L-5291 Bilal Ahmed   Computer Science        sem 3   72.5  B

No match: []


## 7. `update_record()` and `delete_record()`

In [7]:
print(backend.update_record("25L-5291", {"name": "Bilal Ahmed", "department": "Computer Science",
                                         "semester": 3, "marks": 81}))
print(backend.update_record("25L-9999", {"name": "Ghost", "department": "Law", "semester": 1, "marks": 50}))
print(backend.delete_record("25L-5160"))
print(backend.delete_record("25L-5160"))     # already deleted
print("Remaining:", len(backend.load_records()))

(True, ['Updated Bilal Ahmed.'])
(False, ['No student with roll number 25L-9999.'])
(True, ['Deleted 25L-5160.'])
(False, ['No student with roll number 25L-5160.'])
Remaining: 5


## 8. `generate_report()` — statistics

In [8]:
rep = backend.generate_report()
print(f"Students: {rep['total']}   Average: {rep['average']}   Pass rate: {rep['pass_rate']}%")
print("Highest:", rep['highest']['name'], rep['highest']['marks'], "| Lowest:", rep['lowest']['name'], rep['lowest']['marks'])
print("\nGrade distribution")
for grade, count in rep["grades"].items():
    print(f"  {grade}: {'#' * count} ({count})")
print("\nAverage by department")
for dept, info in rep["by_department"].items():
    print(f"  {dept:<26}{info['average']:>5}  ({info['count']} student(s))")

Students: 5   Average: 80.6   Pass rate: 100.0%
Highest: Hira Noor 95.0 | Lowest: Usman Raza 48.0

Grade distribution
  A: ### (3)
  B: # (1)
  C:  (0)
  D: # (1)
  F:  (0)

Average by department
  Computer Science           84.5  (2 student(s))
  Software Engineering       91.0  (1 student(s))
  Electrical Engineering     48.0  (1 student(s))
  Law                        95.0  (1 student(s))


## 9. File persistence — data written to disk
Everything above was saved to JSON. A fresh `load_records()` (as at portal startup) reads it back.

In [9]:
print(sorted(os.listdir(SANDBOX)))
with open(backend.STUDENTS_FILE) as f:
    print(f.read()[:600], "...")
print("\nReloaded from disk:", len(backend.load_records()), "records")

['students.json', 'users.json']
[
  {
    "roll_no": "25L-5308",
    "name": "Ayesha Khan",
    "department": "Computer Science",
    "semester": 3,
    "marks": 88.0,
    "grade": "A"
  },
  {
    "roll_no": "25L-5291",
    "name": "Bilal Ahmed",
    "department": "Computer Science",
    "semester": 3,
    "marks": 81.0,
    "grade": "B"
  },
  {
    "roll_no": "25L-5101",
    "name": "Sara Malik",
    "department": "Software Engineering",
    "semester": 2,
    "marks": 91.0,
    "grade": "A"
  },
  {
    "roll_no": "25L-5133",
    "name": "Usman Raza",
    "department": "Electrical Engineering",
    "semester": 4,
    "ma ...

Reloaded from disk: 5 records


## 10. Robustness — corrupted and missing files
The portal must not crash if the data file is damaged or deleted.

In [10]:
with open(backend.STUDENTS_FILE, "w") as f:
    f.write("{ this is not valid json")
print("Corrupted file ->", backend.load_records())

os.remove(backend.STUDENTS_FILE)
print("Missing file   ->", backend.load_records())

[warning] students.json is corrupted; starting empty.
Corrupted file -> []
Missing file   -> []


## 11. Interactive console menu (`while True`)
`backend.main()` is the terminal version of the portal: it loops until the user picks *Exit*. Here we feed it scripted input to show a sample session.

In [11]:
import builtins, io, contextlib
script = iter(["admin", "admin123",           # login
               "2", "Demo Student", "25L-0001", "Law", "2", "77",   # add
               "1",                           # list
               "6",                           # report
               "7"])                          # exit
real_input = builtins.input
builtins.input = lambda prompt="": (print(prompt, end=""), next(script))[1]
try:
    backend.main()
finally:
    builtins.input = real_input

=== Student Management Portal ===
Username: Password: 
1) List  2) Add  3) Search  4) Update  5) Delete  6) Report  7) Exit
Choose: Name: Roll no: Department: Semester (1-8): Marks (0-100): Added Demo Student (25L-0001).

1) List  2) Add  3) Search  4) Update  5) Delete  6) Report  7) Exit
Choose:   Roll No     Name                    Department              Sem  Marks  Grade
  25L-0001    Demo Student            Law                     2    77.0   B

1) List  2) Add  3) Search  4) Update  5) Delete  6) Report  7) Exit
Choose: Students: 1  Average: 77.0  Pass rate: 100.0%
  A:  (0)
  B: # (1)
  C:  (0)
  D:  (0)
  F:  (0)

1) List  2) Add  3) Search  4) Update  5) Delete  6) Report  7) Exit
Choose: Goodbye.


## 12. Website
The Flask frontend lives in `app.py` (templates in `templates/`, styles in `static/`). Run it from a terminal:

```
pip install flask
python seed_data.py     # optional sample data
python app.py           # open http://127.0.0.1:5000  (admin / admin123)
```
Screenshots of the running website are included in the submission.

In [12]:
shutil.rmtree(SANDBOX, ignore_errors=True)   # tidy up the sandbox
print("Done.")

Done.
